# Using RAG to chat about class notes
### Author: Cole Drumheller

In [ ]:
# imports
import os
import time
from dotenv import load_dotenv
from langchain.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

In [11]:
def loadDocuments():
    # load documents from a directory
    loader = DirectoryLoader('data', glob='**/*.pdf', loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

def createEmbeddingsDB(documents):
    # get embeddings
    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
    chunk = splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings()
    db = Chroma.from_documents(chunk, embeddings, persist_directory='chroma_db1500')
    db.persist()
    return db

def loadDatabase():
    if os.path.exists('chroma_db1500'):
        print("Database exists, loading...")
        embeddings = OpenAIEmbeddings()
        return Chroma(persist_directory='chroma_db1500', embedding_function=embeddings)
    else:
        print("Database does not exist, creating...")
        documents = loadDocuments()
        return createEmbeddingsDB(documents)

In [ ]:
# Main code

# load api key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = api_key

# load/create database
db = loadDatabase()
# set our data as the retriever
retriever = db.as_retriever(search_kwargs={"k": 2})

# initialize LLM and QA chain
llm = ChatOpenAI(model_name='gpt-5-nano', temperature=1)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)

# Main loop
print("Course Assistant:")
print("Type exit to quit")

# loop until user exits
while True:
    # get user input
    query = input("Enter Question: ")
    print("Question:", query)
    # check for exit
    if query.lower() == 'exit':
        break

    # get answer and sources from LLM
    answer = qa_chain(query)
    print("Answer:\n", answer['result'])
    print("Sources:")
    # print source documents
    for i, doc in enumerate(answer["source_documents"], start=1):
            source = os.path.basename(doc.metadata.get("source", "unknown"))
            snippet = doc.page_content[:180].replace("\n", " ")
            print(f"  {i}. {source} — \"{snippet}...\"")
    print()
    # sleep so there's time to show output before next input
    time.sleep(2)

Database does not exist, creating...
Course Assistant:
Type exit to quit
Question: What methods of hyperparameter tuning were discussed?
Answer:
 The methods discussed were:

- Empirical process (manual trial and error)
- Grid search
- Randomized search
- Bayesian optimization (using a Gaussian process surrogate to guide the search)

Notes:
- Grid search is typically best with few hyperparameters and narrow ranges.
- Randomized search can be more effective with many hyperparameters and wider ranges.
- Bayesian optimization uses initial evaluations to build a surrogate model and select the next points to try.
Sources:
  1. Hyperparameter_Note_Framework.pdf — "Hyperparameter Note-Taking Framework Cole Drumheller Wednesday 27th August, 2025 - 00:58 1 Summary The material covered a few different topics, the difference between parameters an..."
  2. Hyperparameter_Note_Framework.pdf — "to Hyperparameter Tuning — Grid Search vs. Randomized Search, 1:30). You then select the model that perfor